In [24]:
import os 
from langchain_core.prompts import ChatMessagePromptTemplate,ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import OllamaLLM
from langchain_core.runnables import RunnablePassthrough,RunnableBranch,Runnable,RunnableParallel
from langchain_core.messages import SystemMessage,HumanMessage
from langchain_core.tools import tool
import re
from typing import List,Dict,Union

In [59]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="minimax-m3:cloud",
    temperature=0
)

In [17]:
@tool
def add_numbers(inputs:str) ->dict:
    """
    Adds a list of number provided in the input string.
    Parameters:
    -inputs (str):
    string, it should contain numbers that can be extracted and summed.
    Returns:
    - dict: A dictionary with a single key "result" containing the sum of the numbers.
    Example Input:
    "Add number 10,20, and 30."
    Example Output:
    {"result":60}
    """
    # Use regular expression to extract all number from input
    numbers=[int(num) for num in re.findall(r'\d+',inputs)]
    result=sum(numbers)
    return {'result':result}




#### The above functions will now act as a tool. You can inspect the tool's schema and other properties using

In [18]:
print(f'{"="*30}Name of the tool {"="*30}\n',add_numbers.name)
print(f'{"="*30}Description of the tool {"="*30}\n',add_numbers.description)
print(f'{"="*30} Args of the tool {"="*30}\n',add_numbers.args)

==============================Name of the tool ==============================
 add_numbers
==============================Description of the tool ==============================
 Adds a list of number provided in the input string.
Parameters:
-inputs (str):
string, it should contain numbers that can be extracted and summed.
Returns:
- dict: A dictionary with a single key "result" containing the sum of the numbers.
Example Input:
"Add number 10,20, and 30."
Example Output:
{"result":60}
============================== Args of the tool ==============================
 {'inputs': {'title': 'Inputs', 'type': 'string'}}


#### You can call the tool using the invoke method

In [35]:
test_input=' 1,29 and 32'
print(add_numbers.invoke(test_input))

{'result': 62}


In [41]:
from langchain_core.tools import Tool

In [64]:
import re
from langchain_core.tools import tool

@tool
def add_numbers(inputs) -> dict:
    """
    Add numbers from either a string or a list.
    """

    if isinstance(inputs, list):
        result = sum(float(x) for x in inputs)

    elif isinstance(inputs, str):
        numbers = [float(num) for num in re.findall(r'\d+(?:\.\d+)?', inputs)]
        result = sum(numbers)

    else:
        raise ValueError("Input must be a string or list.")

    return {"result": result}

In [65]:
add_tool = Tool(
    name='AddTool',
    func=add_numbers.func,
    description='Add a list of numbers and returns the results'
)

print('tool object', add_tool)


tool object name='AddTool' description='Add a list of numbers and returns the results' func=<function add_numbers at 0x000001CF328A9A80>


#### In this example the toll has two inputs : a string containing the number to add, and second boolean input that determines whether to sum the absolute values of those numbers.

In [21]:
@tool
def add_numbers_with_options(numbers:List[float],absolute:bool=False) ->float:
    """
    Adds a list of numbers provided as input.
    
    Parameters:
    -number (List[float]) : A list of number to be summed.
    =absolute (bool) : If True, use the absolute value of the number before summing.
    
    Returns:
    -float: The total sum of the numbers.
    """
    if absolute:
        numbers=[abs(num) for num in numbers]
    return sum(numbers)

In [23]:
print(add_numbers_with_options.invoke({'numbers':[1,2,-3],'absolute':False}))
print(add_numbers_with_options.invoke({'numbers':[1,2,-3],'absolute':True}))

0.0
6.0


#### Improved tool return types with Python typing

In [31]:
@tool
def sum_numbers_with_complex_output(inputs:str) ->Dict[str,Union[float,str]]:
    """
    Extract and sum all the integers and decimal numbers from the inputs strings.
    
    Parameters:
    -input (str) : A string that may contains numeric values.
    
    Returns:
    -dict : A dictionary with the key 'result'. if numbers are found, the value is ther sum(float)
            if no number are found or an error occurs, the value is a corresponding message (str)
    
    Example Input:
    'Add the numbers 10,20.5 and -30'
    
    Example Output:
    {'result':0.5}
    """
    matches=re.findall(r'-?\d+(?:\.\d+)?',inputs)
    if not matches:
        return {'result':'No number found in inputs'}
    try:
        numbers=[float(num) for num in matches]
        total=sum(numbers)
        return {'result':total}
    except Exception as e:
        return {'result':f'Error during summation:{str(e)}'}
        
        
    

In [32]:
print(sum_numbers_with_complex_output.invoke('what is the sum of 33, 32.5 and 30'))

{'result': 95.5}


### `initialize_agent`

When you set up an agent, you're connecting tools and an LLM to work together seamlessly. The agent uses the LLM to understand what needs to be done and decides which tool to use based on the task. Here's a simple overview of the key parts:


#### **Relationship between agent and LLM**
- The **agent** acts as the decision-maker, figuring out which tools to use and when.
- The **LLM** is the reasoning engine. It:
  - Interprets the user's input.
  - Helps the agent make decisions.
  - Generates a response based on the output of the tools.

Think of the agent as the manager assigning tasks and the LLM as the brain solving problems or delegating work.

---

#### **Key parameters of `initialize_agent`**

1. **`tools`**- see above 

2.  **`llm`** see above 

3. **`agent`**:
   - Specifies the reasoning framework for the agent.
   - `"zero-shot-react-description"` enables:
     - **Zero-shot reasoning**: The agent can solve tasks it hasn't seen before by thinking through the problem step by step.
     - **React framework**: A logical loop of:
       - **Reason** → Think about the task.
       - **Act** → Use a tool to perform an action.
       - **Observe** → Check the tool's output.
       - **Plan** → Decide what to do next.

4. **`verbose`**:
   - If `True`, it prints detailed logs of the agent’s thought process.
   - Useful for debugging or understanding how the agent makes decisions.




In [66]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[add_tool],
    system_prompt='You are a helpful assistant',
    
)


response=agent.invoke(
    {'messages':[{'role':'user',"content":"In 2023, the US GDP was approximately $27.72 trillion, while Canada's was around $2.14 trillion and Mexico's was about $1.79 trillion what is the total."}]}
)

print(response["messages"][-1].content)

The total combined GDP of the US, Canada, and Mexico in 2023 was approximately **$31.65 trillion**.

Breakdown:
- 🇺🇸 United States: $27.72 trillion
- 🇨🇦 Canada: $2.14 trillion
- 🇲🇽 Mexico: $1.79 trillion
- **Total: $31.65 trillion**

This combined figure is often referenced in discussions of the USMCA (formerly NAFTA) trade bloc economy.
